# Make Unit Tests for a spell book

This works for some legacy spell conversion
where a difficulty was parsed as an Aspect.

In [12]:
from pathlib import Path
import textwrap

from opend6_tools.magic2 import *
from analyze_magic import all_books

In [13]:
project_base = Path.home() / "Documents/Hobbies/Gaming/OpenD6"

books = all_books(project_base/"world")

system_book/spells/system_spells.py
fantasy_rulebook/spells/fantasy_CANTRIPS.py
fantasy_rulebook/spells/fantasy_ALTERATION.py
fantasy_rulebook/spells/fantasy_APPORTATION.py
fantasy_rulebook/spells/fantasy_CONJURATION.py
fantasy_rulebook/spells/fantasy_DIVINATION.py
magic_guide/spells/Chronomancy.py
magic_guide/spells/Elemental.py
magic_guide/spells/Necromancy.py
magic_guide/spells/Peregrination.py
magic_guide/spells/Photomancy.py
magic_guide/spells/Somniomancy.py
magic_guide/spells/Technomancy.py
magic_guide/spells/Vitomancy.py
magic_guide/spells/Wizardry.py
magic_guide/spells/sample_spells.py
world_book/spells/fw_cantrips.py
world_book/spells/fw_rank2.py
world_book/spells/fw_rank3.py
world_book/spells/fw_rank4.py
opend6_project/opend6_spells.py
fantasy_rulebook/spells/fantasy_miracle_DIVINATION.py
fantasy_rulebook/spells/fantasy_miracle_FAVOR.py
fantasy_rulebook/spells/fantasy_miracle_STRIFE.py


In [14]:
[(name, len(book)) for name, book in books.items()]

[('system_spells', 13),
 ('fantasy_CANTRIPS', 3),
 ('fantasy_ALTERATION', 5),
 ('fantasy_APPORTATION', 3),
 ('fantasy_CONJURATION', 8),
 ('fantasy_DIVINATION', 3),
 ('Chronomancy', 18),
 ('Elemental', 18),
 ('Necromancy', 36),
 ('Peregrination', 27),
 ('Photomancy', 22),
 ('Somniomancy', 25),
 ('Technomancy', 32),
 ('Vitomancy', 24),
 ('Wizardry', 29),
 ('sample_spells', 5),
 ('fw_cantrips', 6),
 ('fw_rank2', 15),
 ('fw_rank3', 16),
 ('fw_rank4', 5),
 ('Project Cantrips', 2),
 ('Project Alteration Spells', 2),
 ('Project Apportation Spells', 2),
 ('Project Conjuration Spells', 6),
 ('Project Divination Spells', 1),
 ('fantasy_miracle_DIVINATION', 2),
 ('fantasy_miracle_FAVOR', 7),
 ('fantasy_miracle_STRIFE', 7)]

## Make Doctest block

In [15]:
def make_spell_doctest(spell_book: list[Spell], book_attr_name: str = "spells") -> None:
    print("__test__ = {")
    for slot, spell in enumerate(spell_book):
        try:
            expected = int(spell.other_aspects["Difficulty"].format)  # type: ignore
        except KeyError:
            # Ugh. Difficulty not included.
            expected = 0
        except ValueError:
            value, *words = spell.other_aspects["Difficulty"].format.split()  # type: ignore
            low, high = value.split("/")
            expected = int(low)
        print(f'    "{spell.name}": ">>> {book_attr_name}[{slot}].difficulty\\n{expected}",')
    print("}")

In [16]:
make_spell_doctest(books["fw_cantrips"])

__test__ = {
    "Protection from Magic": ">>> spells[0].difficulty\n0",
    "Halt": ">>> spells[1].difficulty\n0",
    "Illusion of Rat": ">>> spells[2].difficulty\n0",
    "Communicate w/ Plant/Animal": ">>> spells[3].difficulty\n0",
    "Knock": ">>> spells[4].difficulty\n0",
    "Shocking Grasp": ">>> spells[5].difficulty\n0",
}


## Make pytest block

In [19]:
def make_spell_pytest(spell_book: list[Spell], book_attr_name: str = "spells") -> None:
    print("SPELL_DIFFICULTIES = [")
    for spell in spell_book:
        try:
            expected = int(spell.other_aspects["Difficulty"].format)  # type: ignore
        except KeyError:
            # Ugh. Difficulty not included.
            expected = 0
        except ValueError:
            value, *words = spell.other_aspects["Difficulty"].format.split()  # type: ignore
            low, high = value.split("/")
            expected = int(low)
        print(f'    ("{spell.name}", {expected}),')
    print("]")
    print(
        textwrap.dedent(f"""\
        SPELL_MAP = {{s.name: s for s in {book_attr_name}}}

        @pytest.mark.parametrize("spell_name,difficulty", SPELL_DIFFICULTIES)
        def test_spells(spell_name, difficulty):
            assert SPELL_MAP[spell_name].difficulty == difficulty
    """)
    )

In [20]:
make_spell_pytest(books["fw_cantrips"])


SPELL_DIFFICULTIES = [
    ("Protection from Magic", 0),
    ("Halt", 0),
    ("Illusion of Rat", 0),
    ("Communicate w/ Plant/Animal", 0),
    ("Knock", 0),
    ("Shocking Grasp", 0),
]
SPELL_MAP = {s.name: s for s in spells}

@pytest.mark.parametrize("spell_name,difficulty", SPELL_DIFFICULTIES)
def test_spells(spell_name, difficulty):
    assert SPELL_MAP[spell_name].difficulty == difficulty



In [ ]:
s